# InstanSeg: Custom Dataset Creation, Training, and Testing
This notebook outlines the end-to-end pipeline for utilizing InstanSeg for cell instance segmentation. It covers:
1. Environment Setup & Dependency Installation
2. Data Preprocessing & Custom `.pth` Dataset Creation
3. Model Fine-Tuning & Exporting
4. Testing

*Notebook last updated in May 2026. All new standards, versions, and requirements will need to be handled separately.*

In [ ]:
# Install the custom fork of InstanSeg
!git clone https://github.com/sonyalytv/instanseg_cryobiology.git

In [ ]:
!pip install -e /kaggle/working/instanseg_cryobiology

In [ ]:
# Install necessary external dependencies
!pip install monai
!pip install stardist
!pip install edt
!pip install fastremap

In [ ]:
!pip install --upgrade ipython ipykernel

**Restart the kernel**
* Choose `Run` in the upper menu
* Choose `Restart & clear cell outputs`

In [1]:
import torch
import torchvision
import instanseg

## 1. Dataset Creation

In [ ]:
import os
import torch
import torchvision
from pathlib import Path

# REQUIRED
%load_ext autoreload
%autoreload 2

# Configure InstanSeg environment paths
os.environ['INSTANSEG_RAW_DATASETS'] = os.path.abspath("/kaggle/working/instanseg_cryobiology/Raw_Datasets/")
os.environ['INSTANSEG_DATASET_PATH'] = os.path.abspath("/kaggle/working/instanseg_cryobiology/instanseg/datasets/")

# Create directories if they do not exist
if not os.path.exists(os.environ['INSTANSEG_RAW_DATASETS']):
    os.mkdir(os.environ['INSTANSEG_RAW_DATASETS'])

if not os.path.exists(os.environ['INSTANSEG_DATASET_PATH']):
    os.mkdir(os.environ['INSTANSEG_DATASET_PATH'])

print("Environment paths configured.")

## Use Case 1: Dataset Preparation from Custom Data
This section copies custom source images and masks, ensures proper naming conventions, handles text-to-binary mask conversions (or zero masks), and compiles them into `target_dataset.pth`.

In [ ]:
import shutil

# Setup Mask Directory
mask_directory = Path("/kaggle/working/cell-masks")
mask_directory.mkdir(parents=True, exist_ok=True)
source_masks = "/kaggle/input/datasets/sofiialytvynenko/test-neuro-1"
shutil.copytree(source_masks, str(mask_directory), dirs_exist_ok=True)

# Rename files to include "_masks" suffix if missing
for file in mask_directory.iterdir():
    if file.is_file() and not file.stem.endswith("_masks"):
        new_name = file.stem + "_masks" + file.suffix
        file.rename(file.parent / new_name)

# Setup Images Directory
img_directory = Path("/kaggle/working/cell-images")
img_directory.mkdir(parents=True, exist_ok=True)
source_images = "/kaggle/input/datasets/sofiialytvynenko/test-neuro"
shutil.copytree(source_images, str(img_directory), dirs_exist_ok=True)

# Move everything to the InstanSeg Raw_Datasets folder
raw_data_path = "/kaggle/working/instanseg_cryobiology/Raw_Datasets/my_dataset"
shutil.copytree(str(img_directory), raw_data_path, dirs_exist_ok=True)
shutil.copytree(str(mask_directory), raw_data_path, dirs_exist_ok=True)

### Data Preparation Option A: Generate Blank Masks (For Inference)
Run this cell **only if** your dataset consists purely of raw images without annotations (e.g., you are preparing a dataset strictly for testing/inference). This script will read your images and generate corresponding blank masks required by the InstanSeg dataset builder.

In [ ]:
import numpy as np
import cv2

def zero_mask(image_shape):
    mask = np.zeros(image_shape, dtype=np.uint16)
    return mask

data_path = "/kaggle/working/instanseg_cryobiology/Raw_Datasets/my_dataset"
image_format = ".tif"

for file in sorted(os.listdir(data_path)):
    file = os.path.join(data_path, file)
    name, _ = os.path.splitext(file)
    mask_filename = f"{name}_masks{image_format}"
    output_path = os.path.join(data_path, mask_filename)
    
    image = cv2.imread(file)
    image_shape = image.shape
    mask = zero_mask((image_shape[0], image_shape[1], 1))
    cv2.imwrite(output_path, mask)

### Data Preparation Option B: Convert TXT Annotations to Masks
Run this cell **only if** your dataset comes with `.txt` files containing polygon coordinates (e.g., YOLO format). This script will map those normalized coordinates onto a blank canvas to create the standard 2D instance masks required by InstanSeg.

In [ ]:
import numpy as np
import cv2

def txt_to_binary_mask(txt_file, image_shape):
    """Converts polygon coordinates from a TXT file into a 2D instance mask."""
    mask = np.zeros(image_shape, dtype=np.uint16)
    h, w, c = image_shape
    label = 1
    
    with open(txt_file, 'r') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Extract coordinates (the first value is a class ID and is skipped)
            values = list(map(float, line.split()))
            coords = values[1:]
            polygon = np.array(coords, dtype=np.float32).reshape(-1, 2)

            # Denormalize coordinates from [0, 1] back to absolute pixel values
            polygon[:, 0] *= w
            polygon[:, 1] *= h
            polygon = np.round(polygon).astype(np.int32)

            # Draw the filled polygon with a unique instance label
            cv2.fillPoly(mask, [polygon], color=label)
            label += 1
    # print(label)    
    
    return mask

data_path = "/kaggle/working/instanseg_cryobiology/Raw_Datasets/my_dataset"
image_shape = (1024, 1024, 1) # change if needed
image_format = ".tif"

for file in sorted(os.listdir(data_path)):
    file = os.path.join(data_path, file)
    name, ext = os.path.splitext(file)
    if "masks" in file and ext == ".txt":
        mask = txt_to_binary_mask(file, (image_shape[0], image_shape[1], 1))
        # show_images(image, mask, colorbar=False)
        output_path = os.path.join(data_path, os.path.basename(file).replace(".txt", image_format))
        cv2.imwrite(output_path, mask)
        os.remove(file)

Creating a `.pth` dataset

In [ ]:
import fastremap
import numpy as np
from skimage import io, color
from instanseg.utils.utils import show_images

def load_custom_dataset(dataset_dict):
    data_path = os.path.abspath("/kaggle/working/instanseg_cryobiology/Raw_Datasets/my_dataset")
    if not os.path.exists(data_path):
        raise FileNotFoundError(data_path)
        
    items = []
    for file in sorted(os.listdir(data_path)):
        file_path = os.path.join(data_path, file)
        
        if "masks" in file:
            continue # Skip processing masks directly, handle them via images

        mask_path = str(Path(file_path).parent / Path(file_path).stem) + "_masks" + ".tif"
        image = io.imread(file_path)
        masks = io.imread(mask_path)

        # DO resizing here if needed
        # h, w = image.shape[:2]
        # scale = max(1, w // 512, h // 512)
        # new_size = (w // scale, h // scale)
        # pil_image = Image.fromarray(image)
        # pil_mask = Image.fromarray(masks)
        # image = np.array(pil_image.resize(new_size, Image.LANCZOS))
        # masks = np.array(pil_mask.resize(new_size, Image.NEAREST))

        # Convert grayscale to RGB if necessary
        if len(image.shape) < 3:
            image_array = np.array(image)
            rgb_array = np.stack([image_array] * 3, axis=-1)
            image = np.array(Image.fromarray(rgb_array, mode="RGB"))
        
        # Renumber masks sequentially
        labels, remapping = fastremap.renumber(masks, in_place=True)
        
        item = {
            'cell_masks': fastremap.refit(labels),
            'parent_dataset': "my_dataset", 
            'image': image,
            'original_size': image.shape,
            'image_modality': "Brightfield",
            'file_name': file_path
        }
        items.append(item)

    assert len(items) > 0, "No items found in the dataset folder."

    # Shuffle and split data
    np.random.seed(42)
    np.random.shuffle(items)
    # Note: Adjust the split percentages
    dataset_dict['Train'] += items[:int(len(items) * 0)]
    dataset_dict['Validation'] += items[int(len(items) * 0):int(len(items) * 0)]
    dataset_dict['Test'] += items[int(len(items) * 0.7):]

    return dataset_dict

# Initialize and build
Segmentation_Dataset = {'Train':[], 'Test':[], 'Validation':[]}
Segmentation_Dataset = load_custom_dataset(Segmentation_Dataset)

# Debugging Block
# item = Segmentation_Dataset_Spring['Train'][-1]
# print("Image shape:", item['image'].shape)
# print("Unique mask values:", np.unique(item['cell_masks']))

# show_images(item['image'], item['cell_masks'], colorbar=False)

# print(item['cell_masks'].shape)
# for i in range(item['cell_masks'].shape[0]):
#     for j in range(item['cell_masks'].shape[1]):
#         print(item['cell_masks'][i][j])

In [ ]:
# Save the custom dataset
save_path = os.path.join(os.environ['INSTANSEG_DATASET_PATH'], "target_dataset.pth")
torch.save(Segmentation_Dataset, save_path)
print(f"Spring Dataset saved to {save_path}")

## Use Case 2: Dataset Preparation from LIVECell
This alternative method relies on the LIVECell dataset annotations and compiles them into `livecell_dataset.pth`.

In [ ]:
from PIL import Image

# Dataset splits
splits = {
    "Train": {
        "images": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/train/images",
        "masks": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/train/masks",
        "start": 0,
        "finish": 350
    },
    "Validation": {
        "images": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/train/images",
        "masks": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/train/masks",
        "start": 350,
        "finish": 449
    },
    "Test": {
        "images": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/test/images",
        "masks": "/kaggle/input/datasets/sofiialytvynenko/livecell-shsy5y/LIVECell/test/masks",
        "start": None,
        "finish": None
    },
}

Segmentation_Dataset = {"Train": [], "Validation": [], "Test": []}

def load_image_as_rgb_tiff(image_path):
    img = io.imread(image_path)
    if img.ndim == 2:
        img = np.stack([img]*3, axis=-1)
    img = Image.fromarray(img.astype(np.uint8))
    return np.array(img)

def load_mask_as_tiff(mask_path):
    mask = io.imread(mask_path)
    if mask.ndim == 3:
        mask = mask[..., 0]
    mask = fastremap.refit(mask)
    return mask.astype(np.uint16)

# Build dataset
for split, paths in splits.items():
    img_dir = Path(paths["images"])
    mask_dir = Path(paths["masks"])

    img_files = sorted(img_dir.glob("*.png"))
    if paths["start"] is not None and paths["finish"] is not None:
        img_files = img_files[paths["start"]:paths["finish"]]
        print(f"{split}: Using {len(img_files)} images")

    for img_path in img_files:
        base_name = img_path.stem.replace("_img", "").replace("_masks", "")
        mask_path = mask_dir / f"{base_name}_masks.png"

        if not mask_path.exists():
            print(f"Missing mask for {img_path.name}")
            continue

        image = load_image_as_rgb_tiff(str(img_path))
        mask = load_mask_as_tiff(str(mask_path))
        h, w = image.shape[:2]
        
        item = {
            "image": image,
            "cell_masks": mask,
            "parent_dataset": "my_dataset", 
            "original_size": (h, w),
            "image_modality": "Brightfield",
            "file_name": str(img_path),
        }
        Segmentation_Dataset[split].append(item)

print(len(Segmentation_Dataset["Validation"]))

# Save dataset
livecell_save_path = os.path.join(os.environ['INSTANSEG_DATASET_PATH'], "livecell_dataset.pth")
torch.save(Segmentation_Dataset, livecell_save_path)
print(f"Dataset saved to {livecell_save_path}")

# Quick check
if len(Segmentation_Dataset["Train"]) > 0:
    sample = Segmentation_Dataset["Train"][0]
    print("Sample keys:", sample.keys())
    print("Image shape:", sample["image"].shape)
    print("Mask unique values:", np.unique(sample["cell_masks"])[:10])

## 2. Model Training & Exporting

In this section, we will train our InstanSeg model on the dataset we just created. Training from scratch can take a long time, so it is common practice to start with pre-trained weights (Transfer Learning) or resume from a previous checkpoint. 

First, we will set up the directory for our model and move an existing checkpoint into it (simply skip this part if you want to train from scartch).

In [ ]:
from pathlib import Path
import shutil

model_folder_name = "my_first_instanseg"
model_dir = Path(f"/kaggle/working/instanseg_cryobiology/instanseg/models/{model_folder_name}")
model_dir.mkdir(parents=True, exist_ok=True)

source_weights = "/kaggle/input/models/sofiialytvynenko/instanseg-neuro/pytorch/full-livecell/1/model_weights_best.pth"
destination_weights = model_dir / "model_weights_best.pth"

# Copy the weights into our new model directory so training can resume from them
shutil.copy(source_weights, destination_weights)

### Initiating the Training Process

Now we call the `instanseg_training` function. This function handles the training loop behind the scenes. 
Here is a quick breakdown of the key parameters you might want to tweak:
* `target_segmentation="C"`: Tells the model we are segmenting Cells (use "N" for Nuclei).
* `num_epochs`: The maximum number of passes through the training data.
* `max_no_improvement`: Early stopping criterion. If the model doesn't improve for this many epochs, training stops to prevent overfitting.
* `hotstart_training`: The number of initial epochs to train before evaluating.

In [ ]:
from instanseg.scripts.train import instanseg_training

# Note: Ensure that 'Segmentation_Dataset' is loaded in memory from Section 1
# before running this block.

instanseg_training(
    data_path = "/kaggle/working/instanseg_cryobiology/instanseg/datasets/livecell_dataset.pth",
    segmentation_dataset = Segmentation_Dataset, # The dictionary we built earlier
    source_dataset = "my_dataset",               # The name tagged to our data
    model_folder = "my_first_instanseg",         # Must match the folder created above
    model_path = "/kaggle/working/instanseg_cryobiology/instanseg/models/",
    output_path = "/kaggle/working/instanseg_cryobiology/instanseg/models/",
    experiment_str = "my_first_instanseg",
    requested_pixel_size = 0.5,                  # Adjust based on your microscope resolution
    target_segmentation = "C",                   # 'C' for Cells, 'N' for Nuclei
    num_epochs = 500,                            # Max epochs to run
    max_no_improvement = 5,                      # Early stopping
    hotstart_training = 5
)

print("Training complete!")

### Saving and Exporting the Model

Because Kaggle clears its `/kaggle/working/` directory when the session ends, we need to zip our newly trained model folder so it can be easily downloaded via the Kaggle interface.

We also use `export_to_torchscript`. TorchScript optimizes the PyTorch model, making it faster and allowing it to be deployed in production environments.

In [ ]:
!zip -r my_instanseg.zip "/kaggle/working/instanseg_cryobiology/instanseg/models/my_first_instanseg"

In [2]:
from instanseg.utils.utils import export_to_torchscript
import os

os.environ["INSTANSEG_MODEL_PATH"] = str(os.path.abspath("/kaggle/input/models/sofiialytvynenko/instanseg-neuro/pytorch/full-livecell")) # exclude version folder (the last one)
os.environ["INSTANSEG_TORCHSCRIPT_PATH"] = str(os.path.abspath("/kaggle/working"))

export_to_torchscript("1", show_example=False) # name should be the last folder name in the model path (version)

Generating InstanSeg_UNet


2026-04-27 09:56:32.964989: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777283793.159842     176 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777283793.214227     176 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777283793.669141     176 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777283793.669187     176 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777283793.669190     176 computation_placer.cc:177] computation placer alr

Saved torchscript model to /kaggle/working/1.pt


## 3. Testing

This part is independent from the previous ones. Load your trained model into kaggle: load `experiment_log.csv` and `model_weights_best.pth`. Also load the test dataset in `.pth` format, pay attention to the right splits.

To evaluate our model's performance, we use InstanSeg's built-in testing script. This script calculates metrics and can optionally save the visual overlays of the predictions.

**CLI Parameter Breakdown:**
* `-d_p`: Directory path containing the dataset to test.
* `-m_p`: Path to the trained model directory you want to evaluate.
* `-o_f`: Output folder for the test results.
* `-data`: Name of the specific `.pth` dataset file to evaluate against.
* `-save_ims True`: Saves visual images of the predicted masks overlaid on the input.
* `-target "C"`: Segmenting Cells.
* `-set "Test"`: Specifies which split of the `.pth` file to use (Train/Validation/Test).

In [3]:
# Change directory to where the test script is located
%cd /kaggle/working/instanseg_cryobiology/instanseg/scripts

!python test.py \
    -d_p "/kaggle/input/datasets/sofiialytvynenko/target-cell-dataset" \
    -m_f "" -o_f "/kaggle/working/test_results" \
    -m_p "/kaggle/input/models/sofiialytvynenko/instanseg-neuro/pytorch/full-livecell/1" \
    -save_ims True \
    -data "target_dataset (1).pth" \
    -target "C" \
    -set "Test" \
    -cpu_and_ram True

/kaggle/working/instanseg_cryobiology/instanseg/scripts
2026-04-27 09:57:19.143061: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777283839.170767     230 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777283839.178892     230 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777283839.200311     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777283839.200361     230 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777283839.200368  

In [4]:
# Return to the root working directory
%cd /kaggle/working

# Zip the results folder so it can be downloaded and analyzed locally
!zip -r test_results.zip "/kaggle/working/test_results"

/kaggle/working
  adding: kaggle/working/test_results/ (stored 0%)
  adding: kaggle/working/test_results/images/ (stored 0%)
